In [ ]:
import kagglehub

path = kagglehub.dataset_download("mugheesahmad/sh17-dataset-for-ppe-detection")
print("SH17 downloaded at:", path)

import os
print(f"Images: {len(os.listdir(f'{path}/images'))}")

Using Colab cache for faster access to the 'sh17-dataset-for-ppe-detection' dataset.
SH17 downloaded at: /kaggle/input/sh17-dataset-for-ppe-detection
Images: 8099


In [ ]:
# Install
!pip install ultralytics -q


print("package installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 77.2 MB/s eta 0:00:00
package installed


In [ ]:
import yaml
import os

# Read train/val splits
with open(f'{path}/train_files.txt') as f:
    train_files = [line.strip() for line in f]
with open(f'{path}/val_files.txt') as f:
    val_files = [line.strip() for line in f]

# Write image path lists
with open('/content/train.txt', 'w') as f:
    f.write('\n'.join([f'{path}/images/{fn}' for fn in train_files]))
with open('/content/val.txt', 'w') as f:
    f.write('\n'.join([f'{path}/images/{fn}' for fn in val_files]))

# Create data.yaml
data = {
    'train': '/content/train.txt',
    'val': '/content/val.txt',
    'nc': 17,
    'names': ['person','ear','earmuffs','face','face-guard','face-mask',
              'foot','tool','glasses','gloves','helmet','hands','head',
              'medical-suit','shoes','safety-suit','safety-vest']
}
with open('/content/sh17.yaml', 'w') as f:
    yaml.dump(data, f)

print(f"Train: {len(train_files)} | Val: {len(val_files)} ")

Train: 6479 | Val: 1620 


In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11m.pt')

results = model.train(
    data='/content/sh17.yaml',
    epochs=50,
    imgsz=640,    # back to 640 — better for small objects like face/ear
    batch=16,     # smaller batch for larger image size
    device=0,
    project='/content',
    name='sh17_yolo11m_v2',
    pretrained=True,
    patience=15,
    save=True,
    cache='ram',
    amp=True,
)

print("Training complete")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.106 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=ram, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/sh17.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchs

In [ ]:
import os

# Find best.pt
weights_path = '/content/sh17_yolo11m_v2/weights/best.pt'
size = os.path.getsize(weights_path) / 1e6
print(f"best.pt found: {size:.1f} MB ")

In [ ]:
YOLO_WEIGHTS = "models/best.pt"

In [ ]:
from ultralytics import YOLO

model = YOLO('/content/sh17_yolo11m_v2/weights/best.pt')
results = model.val(
    data='/content/sh17.yaml',
    imgsz=640,
    device=0,
)